In [0]:
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from pyspark.ml.feature import StringIndexer
from pyspark.ml.feature import OneHotEncoder
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder

In [0]:
# Create folder
section = "3"
number = "1"
folder_path = f"dbfs:/student-groups/Group_{section}_{number}"

# Load saved clean data
df_otpw_clean = spark.read.option("header","true").parquet(f"{folder_path}/otpw_clean.parquet").cache()

display(df_otpw_clean.limit(10))

In [0]:
print(len(df_otpw_clean.columns))

In [0]:
print(df_otpw_clean.count())

In [0]:
df_pd = df_otpw_clean.sample(0.05, seed=42).toPandas()

## 0. Data Summary Visualization

In [0]:
flights_month = df_pd.groupby("MONTH").size().reset_index(name="count")
delay_month = df_pd.groupby("MONTH")["DEP_DEL15"].mean().reset_index()
airline_counts = (
    df_pd.groupby("OP_UNIQUE_CARRIER")
    .size()
    .sort_values(ascending=False)
    .head(10)
)
airport_counts = (
    df_pd.groupby("ORIGIN")
    .size()
    .sort_values(ascending=False)
    .head(10)
)

In [0]:
fig, axes = plt.subplots(2,2, figsize=(16,12))

# Flights per month
sns.barplot(
    x="MONTH",
    y="count",
    data=flights_month,
    ax=axes[0,0]
)
axes[0,0].set_title("Flights per Month")

# Delay rate
sns.barplot(
    x="MONTH",
    y="DEP_DEL15",
    data=delay_month,
    ax=axes[0,1]
)
axes[0,1].set_title("Delay Rate per Month")

# Airlines
sns.barplot(
    x=airline_counts.values,
    y=airline_counts.index,
    ax=axes[1,0]
)
axes[1,0].set_title("Top Airlines by Flight Volume")

# Airports
sns.barplot(
    x=airport_counts.values,
    y=airport_counts.index,
    ax=axes[1,1]
)
axes[1,1].set_title("Top Origin Airports")

plt.tight_layout()
plt.show()

## 1. Feature Categorization

In [0]:
df_otpw_clean.columns

In [0]:
['QUARTER',             # temporal features
 'DAY_OF_MONTH',        # temporal features
 'DAY_OF_WEEK',         # temporal features
 'FL_DATE',
 'OP_UNIQUE_CARRIER',   # categorical feature
 'OP_CARRIER',
 'TAIL_NUM',
 'OP_CARRIER_FL_NUM',
 'ORIGIN',              # categorical feature
 'ORIGIN_CITY_NAME',
 'ORIGIN_STATE_ABR',
 'ORIGIN_STATE_FIPS',
 'ORIGIN_STATE_NM',
 'DEST',                # categorical feature
 'DEST_CITY_NAME',
 'DEST_STATE_ABR',
 'DEST_STATE_FIPS',
 'DEST_STATE_NM',
 'CRS_DEP_TIME',        # temporal features
 'DEP_TIME',            # may be leaky
 'DEP_DEL15',
 'DEP_TIME_BLK',
 'CRS_ARR_TIME',        # temporal feature
 'CANCELLED',           # categorical feature
 'DIVERTED',            # categorical feature
 'CRS_ELAPSED_TIME',
 'DISTANCE',
 'DISTANCE_GROUP',
 'YEAR',
 'MONTH',               # categorical feature
 'origin_airport_name',
 'origin_station_name',
 'origin_station_id',
 'origin_iata_code',
 'origin_icao',
 'origin_type',
 'origin_region',
 'origin_station_lat',      # route features
 'origin_station_lon',
 'origin_airport_lat',      # route features
 'origin_airport_lon',
 'origin_station_dis',
 'dest_airport_name',
 'dest_station_name',
 'dest_station_id',
 'dest_iata_code',
 'dest_icao',
 'dest_type',
 'dest_region',
 'dest_station_lat',    # route features
 'dest_station_lon',
 'dest_airport_lat',    # route features
 'dest_airport_lon',
 'dest_station_dis',
 'STATION',
 'DATE',
 'LATITUDE',
 'LONGITUDE',
 'ELEVATION',
 'NAME',
 'REPORT_TYPE',
 'SOURCE',
 'HourlyAltimeterSetting',      # weather features
 'HourlyDewPointTemperature',   # weather features
 'HourlyDryBulbTemperature',    # weather features
 'HourlyPrecipitation',         # weather features
 'HourlyPresentWeatherType',    # weather features
 'HourlyPressureTendency',      # weather features
 'HourlyRelativeHumidity',      # weather features
 'HourlySkyConditions',         # weather features
 'HourlySeaLevelPressure',      # weather features
 'HourlyStationPressure',       # weather features
 'HourlyVisibility',            # weather features
 'HourlyWetBulbTemperature',    # weather features
 'HourlyWindDirection',         # weather features
 'HourlyWindGustSpeed',         # weather features
 'HourlyWindSpeed',             # weather features
 'Sunrise',
 'Sunset',
 'DailyAverageDewPointTemperature',
 'DailyAverageDryBulbTemperature',
 'DailyAverageRelativeHumidity',
 'DailyAverageSeaLevelPressure',
 'DailyAverageStationPressure',
 'DailyAverageWetBulbTemperature',
 'DailyAverageWindSpeed',
 'DailyCoolingDegreeDays',
 'DailyDepartureFromNormalAverageTemperature',
 'DailyHeatingDegreeDays',
 'DailyMaximumDryBulbTemperature',
 'DailyMinimumDryBulbTemperature',
 'DailyPeakWindDirection',
 'DailyPeakWindSpeed',
 'DailySustainedWindDirection',
 'DailySustainedWindSpeed',
 'REM',
 'BackupDirection',
 'BackupDistance',
 'BackupDistanceUnit',
 'BackupElements',
 'BackupElevation',
 'BackupEquipment',
 'BackupLatitude',
 'BackupLongitude',
 'BackupName',
 'WindEquipmentChangeDate',
 'flight_id',                   
 'dep_hour',                    # Derived features
 'route',                       # Derived features
 'wind_bin',                    # Derived features
 'distance_bin']                # Derived features


## 2. Feature Selection

In [0]:
# Leakage features - would this be known 2 hours befire departure? if no, its' leakage column
temporal_features = [
    'QUARTER',             # temporal features
    'MONTH',
    'DAY_OF_MONTH',        # temporal features
    'DAY_OF_WEEK',         # temporal features
    'CRS_DEP_TIME',
    'CRS_ARR_TIME'
]

categorical_features = [
    'OP_UNIQUE_CARRIER',   # categorical feature
    'ORIGIN',              # categorical feature
    'DEST',
    'CANCELLED',           # categorical feature
    'DIVERTED',           # categorical feature
    'route'
]

route_features = [
    'origin_station_lat',
    'origin_airport_lat',
    'dest_station_lat',
    'dest_airport_lat'
]

weather_features = [
    'HourlyAltimeterSetting',      # weather features
    'HourlyDewPointTemperature',   # weather features
    'HourlyDryBulbTemperature',    # weather features
    'HourlyPrecipitation',         # weather features
    # 'HourlyPresentWeatherType',    # weather features
    'HourlyPressureTendency',      # weather features
    'HourlyRelativeHumidity',      # weather features
    # 'HourlySkyConditions',         # weather features
    'HourlySeaLevelPressure',      # weather features
    'HourlyStationPressure',       # weather features
    'HourlyVisibility',            # weather features
    'HourlyWetBulbTemperature',    # weather features
    'HourlyWindDirection',         # weather features
    'HourlyWindGustSpeed',         # weather features
    'HourlyWindSpeed',             # weather features
]

additional_features = [
    'dep_hour',
    'wind_bin',
    'distance_bin'
]

# Target features
target_features = ["DEP_DEL15"]

features = temporal_features + categorical_features + weather_features + route_features + additional_features + target_features

df_otpw_features = df_otpw_clean.select(features).cache()

display(df_otpw_features.limit(20))

In [0]:
def is_numeric_type(spark_type):
    # Common numeric types in PySpark
    numeric_types = ['byte', 'short', 'int', 'long', 'float', 'double', 'decimal']
    return any(spark_type.startswith(t) for t in numeric_types)

numeric_cols = []
categorical_cols = []
for col_name in df_otpw_clean.columns:
    col_type = [dtype for name, dtype in df_otpw_clean.dtypes if name == col_name][0]

    if is_numeric_type(col_type):
        numeric_cols.append(col_name)
    else:
        categorical_cols.append(col_name)

print(numeric_cols)
print(categorical_cols)

In [0]:
# Correlation matrix for selected columns

df_sample_pd = df_otpw_clean.sample(0.5, seed=42).toPandas()
corr = df_sample_pd[numeric_cols].corr()

plt.figure(figsize=(12,10))
sns.heatmap(corr, cmap="coolwarm", center=0)

In [0]:
total_rows = df_otpw_clean.count()
categorical_cols = [col for col in categorical_cols if col not in ['flight_id', 'DEP_TIME']]
cardinality_df = df_otpw_clean.select([
    (F.approx_count_distinct(c) / F.lit(total_rows)).alias(c)
    for c in categorical_cols
])

cardinality_df = cardinality_df.select(
    F.explode(
        F.array([
            F.struct(F.lit(c).alias("column"), F.col(c).alias("cardinality_ratio"))
            for c in cardinality_df.columns
        ])
    )
).select("col.*")

# cardinality_df.orderBy(F.desc("cardinality_ratio")).display()

cardinality_pd = cardinality_df.toPandas()
plt.hist(cardinality_pd["cardinality_ratio"], bins=30)
plt.xlabel("Cardinality Ratio")
plt.ylabel("Number of Columns")
plt.title("Distribution of Column Cardinality")

## 3. Feature Engineering


In [0]:
double_expr = "^[-+]?[0-9]*\\.?[0-9]+([eE][-+]?[0-9]+)?$"
int_expr = "^[-+]?[0-9]+$"
date_expr = "^[0-9]{4}-[0-9]{2}-[0-9]{2}$"

def try_cast(df, col_name, target_type, default_value=None):
    """
    Cast column safely. If cast fails, replace with default_value.
    
    Parameters
    ----------
    df : pyspark.sql.DataFrame
    col_name : str
    target_type : str  ('int','double','timestamp','date','string')
    default_value : value used if cast fails
    """
    if target_type == 'int':
        casted_col = F.when(F.col(col_name).rlike(int_expr), F.col(col_name).cast("decimal(38,0)")).otherwise(F.lit(default_value).cast("int")) 
        casted_col = F.when(casted_col.isNotNull(), F.when(casted_col.between(-2147483648, 2147483647), casted_col.cast("int")).otherwise(casted_col.cast('long'))).otherwise(F.lit(default_value).cast("int"))
    elif target_type == 'double':
        casted_col = F.when(F.col(col_name).rlike(double_expr), F.col(col_name).cast("decimal(38,0)")).otherwise(F.lit(default_value).cast("double")) 
        casted_col = F.when(casted_col.isNotNull(), F.when(casted_col.between(-9223372036854775808, 9223372036854775807), casted_col.cast("double")).otherwise(casted_col.cast('decimal(38,0)'))).otherwise(F.lit(default_value).cast("double"))
    else:
        casted_col = F.col(col_name).cast(target_type)

    return df.withColumn(
        col_name,
        F.when(casted_col.isNotNull(), casted_col)
         .otherwise(F.lit(default_value).cast(target_type))
    )

In [0]:
df_otpw_features = try_cast(df_otpw_features, 'HourlyPrecipitation', "double", 0.0)

df_otpw_features.select('HourlyPrecipitation').distinct().show()

In [0]:
# Check if any column has missing values
total_rows = df_otpw_features.count()
missing_df = df_otpw_features.select([
    (F.sum(F.col(c).isNull().cast("int")) / total_rows * 100).alias(c)
    for c in df_otpw_features.columns
])

missing_percent_df = missing_df.select(
    F.explode(
        F.array([
            F.struct(F.lit(c).alias("column"),
                     F.col(c).alias("missing_percent"))
            for c in missing_df.columns
        ])
    ).alias("col")
).select("col.*")

missing_percent_df = missing_percent_df.filter(F.col("missing_percent") > 0.0)

missing_percent_df.orderBy(F.desc("missing_percent")).show(200, False)

In [0]:
# Convert scheduled time to cyclical features
df_otpw_features = df_otpw_features.withColumn(
    "crs_dep_minutes",
    F.floor(F.col("CRS_DEP_TIME") / 100) * 60 + (F.col("CRS_DEP_TIME") % 100)
)

df_otpw_features = df_otpw_features.withColumn("dep_sin", F.sin(2 * 3.1416 * F.col("crs_dep_minutes") / 1440))
df_otpw_features = df_otpw_features.withColumn("dep_cos", F.cos(2 * 3.1416 * F.col("crs_dep_minutes") / 1440))

In [0]:
# Create route delay rate
route_delay_df = df_otpw_features.groupBy("route").agg(
    F.mean("DEP_DEL15").alias("route_delay_rate")
)

df_otpw_features = df_otpw_features.join(route_delay_df, "route", "left")

In [0]:
df_otpw_features = df_otpw_features.drop('route_delay_rate')

In [0]:
df_otpw_features.select('route_delay_rate').show()

In [0]:
# Create airport delay rate features

origin_delay_df = df_otpw_features.groupBy("ORIGIN").agg(
    F.mean("DEP_DEL15").alias("origin_delay_rate")
)

dest_delay_df = df_otpw_features.groupBy("DEST").agg(
    F.mean("DEP_DEL15").alias("dest_delay_rate")
)

df_otpw_features = df_otpw_features.join(origin_delay_df, "ORIGIN", "left")
df_otpw_features = df_otpw_features.join(dest_delay_df, "DEST", "left")

In [0]:
df_otpw_features = df_otpw_features.drop('origin_delay_rate', 'dest_delay_rate')

In [0]:
df_otpw_features.select('origin_delay_rate', 'dest_delay_rate').show()

In [0]:
# bad weather columns
bad_weather_columns = [
    "HourlyVisibility",
    "HourlyWindSpeed",
    "HourlyPrecipitation"
]

df_otpw_features.select(*bad_weather_columns).describe().show()

In [0]:
# Create 'bad weather' feature
df_otpw_features = df_otpw_features.withColumn(
    "bad_weather",
    (F.col("HourlyVisibility") < 3) |
    (F.col("HourlyWindSpeed") > 25) |
    (F.col("HourlyPrecipitation") > 0.1)
)

In [0]:
# Add an interaction feature for wind speed and visibility
df_otpw_features = df_otpw_features.withColumn(
    "wind_visibility_interaction",
    F.col("HourlyWindSpeed") * F.col("HourlyVisibility")
)

In [0]:
distance_delay_rate = (
    df_otpw_features
    .groupBy("distance_bin").agg(F.mean("DEP_DEL15").alias("distance_rate"))
    .toPandas()
)

wind_delay_rate = (
    df_otpw_features
    .groupBy("wind_bin").agg(F.mean("DEP_DEL15").alias("wind_rate"))
    .toPandas()
)

bad_weather_delay_rate = (
    df_otpw_features
    .groupBy("bad_weather").agg(F.mean("DEP_DEL15").alias("bad_weather_rate"))
    .toPandas()
)



In [0]:
fig, axes = plt.subplots(1,3, figsize=(18,5))

# Delay rate by distance_bin
sns.lineplot(
    x="distance_bin",
    y="distance_rate",
    data=distance_delay_rate,
    marker="o",
    ax=axes[0]
)

axes[0].set_title("Delay Rate by Distance Bin")
axes[0].set_xlabel("Distance Bin")
axes[0].set_ylabel("Delay Rate")

# Delay rate by day of wind_bin
sns.barplot(
    x="wind_bin",
    y="wind_rate",
    data=wind_delay_rate,
    ax=axes[1]
)

axes[1].set_title("Delay Rate by Wind Bin")
axes[1].set_xlabel("Wind Bin")
axes[1].set_ylabel("Delay Rate")

# delay by bad weather
sns.barplot(
    x="bad_weather",
    y="bad_weather_rate",
    data=bad_weather_delay_rate,
    ax=axes[2]
)

axes[2].set_title("Delay Rate by Bad Weather")
axes[2].set_xlabel("Bad Weather")
axes[2].set_ylabel("Delay Rate")

plt.tight_layout()
plt.show()

## 4. Visualizations

In [0]:
df_sample_pd = df_otpw_features.sample(0.1, seed=42).toPandas()

In [0]:
# Airline delay by rate
sns.barplot(
    x="OP_UNIQUE_CARRIER",
    y="DEP_DEL15",
    data=df_sample_pd
)

In [0]:
# airport congestion
airport_stats = df_sample_pd.groupby("ORIGIN").agg(
    delay_rate=("DEP_DEL15","mean"),
    flights=("DEP_DEL15","count")
)

sns.scatterplot(data=airport_stats, x="flights", y="delay_rate")

In [0]:
sns.boxplot(
    x="DEP_DEL15",
    y="HourlyWindSpeed",
    data=df_sample_pd
)

## 5. Baseline Model Pipeline

In [0]:
display(df_otpw_features.groupBy("MONTH").count())

In [0]:
# time-based split
train_df = df_otpw_features.filter(F.col("MONTH").isin([1,2]))
test_df = df_otpw_features.filter(F.col("MONTH") == 3)

In [0]:
# Encode categorical features
indexers = [
    StringIndexer(inputCol=c, outputCol=c+"_idx", handleInvalid="keep")
    for c in categorical_features
]

encoder = OneHotEncoder(
    inputCols=[c+"_idx" for c in categorical_features],
    outputCols=[c+"_vec" for c in categorical_features]
)

In [0]:
# Assemble features
feature_cols = (
    temporal_features +
    route_features +
    weather_features +
    additional_features +
    ['crs_dep_minutes', 'dep_sin', 'dep_cos', 'route_delay_rate', 'origin_delay_rate', 'dest_delay_rate', 'bad_weather', 'wind_visibility_interaction'] +
    [c+"_vec" for c in categorical_features]
)

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

In [0]:
feature_cols

In [0]:
# Create regression model
lr = LogisticRegression(
    featuresCol="features",
    labelCol="DEP_DEL15",
    maxIter=20
)

In [0]:
train_df.select(*train_df.columns).show()

In [0]:
# Build pipeline and train model
pipeline = Pipeline(stages=indexers + [encoder, assembler, lr])

model = pipeline.fit(train_df)

In [0]:
# Run the model
pred = model.transform(test_df)
pred.groupBy("DEP_DEL15", "prediction").count().show()

In [0]:
# Evaluate the model
evaluator = BinaryClassificationEvaluator(
    labelCol="DEP_DEL15",
    metricName="areaUnderROC"
)

auc_score = evaluator.evaluate(pred)
print("AUC:", auc_score)

In [0]:
accuracy = MulticlassClassificationEvaluator(
    labelCol="DEP_DEL15",
    metricName="accuracy"
).evaluate(pred)

accuracy = evaluator.evaluate(pred)
print("Test Accuracy:", accuracy)

In [0]:
# Address class imbalance
delay_rate = train_df.select(F.mean("DEP_DEL15")).collect()[0][0]

weighted_train_df = train_df.withColumn(
    "class_weight",
    F.when(F.col("DEP_DEL15") == 1, 1-delay_rate).otherwise(delay_rate)
)

lr = LogisticRegression(
        featuresCol="features",
        labelCol="DEP_DEL15",
        weightCol="class_weight",
        maxIter=20
    )

In [0]:
# Try with balanced lr model
pipeline = Pipeline(stages=indexers + [encoder, assembler, lr])
model = pipeline.fit(weighted_train_df)
pred = model.transform(test_df)
pred.groupBy("DEP_DEL15", "prediction").count().show()

In [0]:
from pyspark.ml.functions import vector_to_array

pred_df = pred.select(F.col("DEP_DEL15"), vector_to_array("rawPrediction")[1].alias("prediction"))
pred_df.show(5)

In [0]:
accuracy = MulticlassClassificationEvaluator(
    labelCol="DEP_DEL15",
    metricName="accuracy"
).evaluate(pred)

accuracy = evaluator.evaluate(pred)
print("Test Accuracy with balanced training data:", accuracy)

In [0]:
# pred_scores = pred.select(
#     F.col("DEP_DEL15"),
#     F.col("probability").getItem(1).alias("score")   # probability of class=1
# )
pred_pd = pred_df.toPandas()

In [0]:
from sklearn.metrics import roc_curve, auc

y_true = pred_pd["DEP_DEL15"]
y_score = pred_pd["prediction"]

fpr, tpr, _ = roc_curve(y_true, y_score)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(7,6))

plt.plot(fpr, tpr, label=f"ROC Curve (AUC = {roc_auc:.3f})")
plt.plot([0,1], [0,1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Baseline Logistic Regression Performance")

plt.legend(loc="lower right")

# annotate accuracy
plt.text(
    0.6,
    0.2,
    "Test Accuracy = 68.83%",
    bbox=dict(facecolor="white", alpha=0.8)
)

plt.show()